# Session 1: the heart is a pump, the vessels are pipes
**A. Real-world problem.** Your heart does not "squirt" blood through the body: it pushes
it round a closed loop of tubes, like a pump in a central-heating system. Why is the
pressure highest just after the heart and lowest just before it? And why does narrowing
the small vessels make the pump work harder? Today we build the simplest model of the loop.

*This is general education. It is not medical advice and the model is not a diagnostic
tool: never use it to judge anyone's health.*

**Plan.** (1) the loop: heart, arteries, small vessels (arterioles and capillaries), veins,
(2) flow = pressure difference / resistance, (3) the pressure falls along the loop,
(4) narrow the small vessels and see what changes.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

TPR_TYPICAL = 18.6          # mmHg per (L/min): a typical textbook total resistance (93 / 5)
SHARES = {"arteries": 0.10, "small vessels": 0.80, "veins": 0.10}   # illustrative split of the resistance

## Step 1: flow = pressure difference / resistance
A pump pushes with a pressure difference `dP` (in mmHg, millimetres of mercury, the unit
doctors use for blood pressure). The pipes resist with a resistance `R` (mmHg per L/min).
The flow `Q` (litres per minute) is

    Q = dP / R

More push gives more flow; more resistance gives less flow. It is Ohm's law for pipes.
Typical textbook numbers: a push of 93 mmHg through 18.6 mmHg per L/min gives 5 L/min.

In [ ]:
def flow(dP, R):
    """Flow in L/min for a pressure difference dP (mmHg) and resistance R (mmHg per L/min)."""
    if R <= 0:
        raise ValueError("resistance must be positive")
    return dP / R

## Step 2: the loop has three kinds of pipe in a row
Blood leaves the heart into the arteries, passes the small vessels (arterioles and
capillaries, where it feeds the tissues) and returns in the veins. Pipes in a row add their
resistances. We use an *illustrative* split: 10% in the arteries, 80% in the small vessels
and 10% in the veins. `narrow` multiplies the resistance of the small vessels only
(1.0 = normal, 1.5 = 50% more resistance).

In [ ]:
def segment_resistances(narrow=1.0, tpr=TPR_TYPICAL):
    """Resistance of arteries, small vessels and veins, in mmHg per (L/min)."""
    return np.array([SHARES["arteries"] * tpr,
                     SHARES["small vessels"] * tpr * narrow,
                     SHARES["veins"] * tpr])


def total_resistance(narrow=1.0, tpr=TPR_TYPICAL):
    return float(segment_resistances(narrow, tpr).sum())

## Step 3: the pressure falls along the loop
The same flow passes every segment, so the pressure lost in a segment is `Q * R_segment`.
Starting at the pump outlet (pressure `dP` above the return pressure, which we take as 0)
the pressure steps down: a little in the arteries, a lot in the small vessels, a little in
the veins, and reaches 0 back at the heart.

In [ ]:
def pressure_profile(dP, narrow=1.0, tpr=TPR_TYPICAL):
    """Pressure (mmHg) at the start of each segment and at the end of the loop: 4 numbers."""
    R = segment_resistances(narrow, tpr)
    Q = flow(dP, R.sum())
    return np.concatenate([[dP], dP - np.cumsum(Q * R)])


def push_needed(Q_target, narrow=1.0, tpr=TPR_TYPICAL):
    """Pressure difference (mmHg) the pump must give to deliver Q_target (L/min)."""
    return Q_target * total_resistance(narrow, tpr)

## Step 4: see it
Left: the pressure along the loop for normal and for narrowed small vessels. Right: the
flow delivered by the same pump as the small vessels narrow.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("normal: flow", round(flow(93, total_resistance()), 2), "L/min")
    print("narrow 1.5, same pump:", round(flow(93, total_resistance(1.5)), 3), "L/min")
    print("pump needed for 5 L/min at narrow 1.5:", round(push_needed(5.0, 1.5), 1), "mmHg")
    xs = [0, 1, 2, 3]
    fig, ax = plt.subplots(figsize=(6, 4))
    for narrow, col in ((1.0, "tab:blue"), (1.5, "tab:red")):
        ax.plot(xs, pressure_profile(93, narrow), "o-", color=col, label=f"narrowing x{narrow}")
    ax.set_xticks(xs)
    ax.set_xticklabels(["heart outlet", "after arteries", "after small vessels", "back at heart"], fontsize=8)
    ax.set_ylabel("pressure above return (mmHg)")
    ax.set_title("Pressure falls along the loop (pump pressure 93 mmHg)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "pressure_along_loop.png", dpi=120)

    n = np.linspace(1.0, 2.0, 41)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(n, [flow(93, total_resistance(k)) for k in n])
    ax2.axhline(5.0, color="gray", ls=":")
    ax2.set_xlabel("narrowing factor of the small vessels (1 = normal)")
    ax2.set_ylabel("flow (L/min)")
    ax2.set_title("Same pump, narrower small vessels: less flow")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "flow_vs_narrowing.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py` print the flow for narrowing 1.0, 1.25 and 1.5 at a pump pressure of 93 mmHg. Predict the three numbers before you run it.
2. **Stretch.** Write `max_narrowing(push, q_min)` that returns the largest narrowing for which the flow stays at least q_min, and use it to check the answer of question type D in Section F.
3. **Explain.** Explain in your own words why the pressure at the end of the loop is 0 in this model and what that 0 means.

## Assessment
- **A. Conceptual.** Why is the pressure highest at the heart outlet and lowest just before the heart, and why do the small vessels cause the largest drop?
- **B. Numerical.** A pump gives only 60 mmHg through the typical resistance of 18.6 mmHg per L/min. What is the flow? Is the answer sensible?
- **C. Graph and simulation interpretation.** In the pressure graph with narrowing 1.5 and a 93 mmHg pump, how much pressure do the small vessels use up, and how does that compare with normal vessels?
- **D. Design and optimisation challenge.** Design: the pump can give at most 120 mmHg and the flow must stay at or above 4.5 L/min. How much can the small vessels narrow (to 2 decimal places)? State the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: 93 mmHg through 18.6 mmHg per L/min gives 5.0 L/min",
                   abs(L.flow(93.0, 18.6) - 5.0) < 1e-9))
    checks.append(("total resistance is 18.6 when nothing is narrowed", abs(L.total_resistance(1.0) - 18.6) < 1e-9))
    checks.append(("worked example: narrowing x1.5 gives total resistance 26.04", abs(L.total_resistance(1.5) - 26.04) < 1e-9))
    q15 = L.flow(93.0, L.total_resistance(1.5))
    checks.append(("worked example: same pump, narrowing x1.5 gives 3.57 L/min", abs(q15 - 3.571) < 1e-3))
    checks.append(("restoring 5 L/min at narrowing x1.5 needs 130.2 mmHg", abs(L.push_needed(5.0, 1.5) - 130.2) < 1e-6))
    prof = L.pressure_profile(93.0)
    checks.append(("the pressure ends at 0 back at the heart", abs(prof[-1]) < 1e-9))
    checks.append(("the pressure never rises along the loop", bool(np.all(np.diff(prof) <= 1e-12))))
    checks.append(("the small vessels lose 74.4 mmHg of the 93", abs((prof[1] - prof[2]) - 74.4) < 1e-9))
    p15 = L.pressure_profile(93.0, 1.5)
    share = (p15[1] - p15[2]) / 93.0
    checks.append(("narrowing x1.5 makes the small vessels take 85.7% of the pressure loss", abs(share - 1.2 / 1.4) < 1e-9))
    checks.append(("doubling the push doubles the flow", abs(L.flow(186.0, 18.6) - 2 * L.flow(93.0, 18.6)) < 1e-9))
    checks.append(("flow is 0 with no push", L.flow(0.0, 18.6) == 0.0))
    try:
        L.flow(93.0, 0.0)
        bad = False
    except ValueError:
        bad = True
    checks.append(("zero resistance is rejected instead of dividing by zero", bad))

    # --- numbers used in the text
    checks.append(("worked example: the segments are 1.86, 14.88 and 1.86 and the small vessels lose 5.0 x 14.88 = 74.4",
                   np.allclose(L.segment_resistances(), [1.86, 14.88, 1.86]) and abs(5.0 * 14.88 - 74.4) < 1e-9))
    checks.append(("worked example: narrowed small vessels have 22.32", abs(L.segment_resistances(1.5)[1] - 22.32) < 1e-9))
    checks.append(("profile at 93 mmHg is 93, 83.7, 9.3, 0", np.allclose(prof, [93.0, 83.7, 9.3, 0.0], atol=1e-9)))
    checks.append(("doubling the small-vessel resistance cuts the flow to 2.78 L/min", abs(L.flow(93.0, L.total_resistance(2.0)) - 2.778) < 1e-3))
    checks.append(("60 mmHg through 18.6 gives 3.23 L/min", abs(L.flow(60.0, 18.6) - 3.2258) < 1e-3))
    checks.append(("narrowing x1.5: the small vessels lose 79.7 mmHg", abs((p15[1] - p15[2]) - 79.714) < 1e-3))
    ok_d = (abs(120.0 / 4.5 - 26.667) < 1e-3 and abs((26.667 / 18.6 - 0.2) / 0.8 - 1.5421) < 1e-3
            and L.flow(120.0, L.total_resistance(1.54)) >= 4.5 > L.flow(120.0, L.total_resistance(1.55))
            and abs(L.flow(120.0, L.total_resistance(1.54)) - 4.505) < 1e-3 and abs(L.flow(120.0, L.total_resistance(1.55)) - 4.480) < 1e-3)
    checks.append(("design: with 120 mmHg and a flow of at least 4.5 L/min the narrowing can reach 1.54 but not 1.55", bool(ok_d)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
